# Predict SWE from MACAv2 Projections using ML Models

This notebook will generate projections of SWE from PRISM and MACA datasets, utilizing the machine learning models trained in the previous notebook. Then, it will perform plotting and analysis of the data.

## Step 1: Set up notebook

In [2]:
# import libraries

# file management
import gc
import joblib
import os
import pathlib
from pathlib import Path
import pyarrow.parquet as pq

# datatypes
import numpy as np
import pandas as pd
import xarray as xr

# geospatial data
import geopandas as gpd
import rioxarray as rxr

# machine learning
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

# plotting
import holoviews as hv
import hvplot.pandas
import hvplot.xarray
import matplotlib
import matplotlib.pyplot as plt
import panel as pn

# misc
import math

In [3]:
# set directories
proj_dir = os.path.join(pathlib.Path.home(),
                        'Documents',
                        'Graduate_School',
                        'EDA_Certificate', 
                        'Summer', 
                        'snow-drought-modeling')

raw_data_dir = os.path.join(proj_dir, 'data', 'raw')
cleaned_data_dir = os.path.join(proj_dir, 'data', 'cleaned')

# sub dirs
data_dir = Path(cleaned_data_dir, 'machine_learning')
maca_dir = Path(data_dir, 'maca')
ml_dir = Path(proj_dir, 'trained_models')

# maca paths
models = {'CNRM-CM5', 'GFDL-ESM2M', 'CanESM2', 'IPSL-CM5A-MR'}
model_paths = {}
for model in models:
    model_paths[model] = Path(maca_dir, f'{model}_ML_features.parquet')

# prism path
prism_path = Path(data_dir, 'prism_raster.parquet')

# ml model paths
rf_path = Path(ml_dir, 'random_forests_swe_model.joblib')
xgb_path = Path(ml_dir, 'xgboost_swe_model.joblib')
lgb_path = Path(ml_dir, 'lightgbm_swe_model.joblib')
ml_model_paths = {
    'Random_Forests': rf_path,
    'XGBoost': xgb_path,
    'LightGBM': lgb_path
}
feat_nm_path = Path(ml_dir, 'ML_feature_names.joblib')

# outputs
output_dir = Path(proj_dir, 'outputs')

## Step 2: Load Data and Models

In [4]:
# load prism and maca dfs
# stream w/ parquet: this lets the computer only read the data it needs at a given instant

# prism data
if os.path.exists(prism_path):
    prism_pq = pq.ParquetFile(prism_path)
    print('Prism data successfully imported!')
else:
    print(f'Prism data not found. Check {data_dir} for file.')

# maca data
maca_pqs = {}
for model in models:
    if os.path.exists(model_paths[model]):
        maca_pqs[model] = pq.ParquetFile(model_paths[model])
        print(f'{model} data successfully loaded!')
    else:
        print(f'{model} file not found; data has not been loaded. Check {maca_dir} for file.')

Prism data successfully imported!
IPSL-CM5A-MR data successfully loaded!
GFDL-ESM2M data successfully loaded!
CNRM-CM5 data successfully loaded!
CanESM2 data successfully loaded!


In [5]:
# load ML models

ml_models = {}
for model in ml_model_paths:
    if os.path.exists(ml_model_paths[model]):
        ml_models[model] = joblib.load(ml_model_paths[model])
        print(f'{model} has been successfully loaded!')
    else:
        print(f'{model} has not been loaded. Check {ml_dir} for file.')

# load feature names
if os.path.exists(feat_nm_path):
    feature_names = joblib.load(feat_nm_path)
    print('Feature Names have been successfully loaded!')
else:
    print(f'Feature Names have not been loaded. Check {ml_dir} for file.')

Random_Forests has been successfully loaded!
XGBoost has been successfully loaded!
LightGBM has been successfully loaded!
Feature Names have been successfully loaded!


## Step 3: Predict SWE across the watershed for PRISM and MACA data

In [6]:
# define a helper function to help manage batching of SWE prediction

def _get_dataset_batches(data, batch_size):
    '''
    Generates batch indicies and sizes for DataFrames or ParquetFiles iteratively to use ram efficiently.

    Args:
    -----
    data (pd.DataFrame or pd.ParquetFile):
        Input data to be split into batches for ML prediction.
    batch_size (int):
        Number of rows to stream and predict per batch. Default is 500,000, but can be changed depending on system RAM

    Returns:
    --------
    i (int):
        Generation counter
    total_batches (int):
        Total number of batches calculated.
    df_chunk (pd.DataFrame):
        Current chunk of the dataset in DataFrame format.
    '''

    # if data are in a ParquetFile, use this code
    if isinstance(data, pq.ParquetFile):

        # get number of rows
        total_rows = data.metadata.num_rows

        # get number of batches
        total_batches = math.ceil(total_rows / batch_size)

        # generate next i, total_batches, and convert next batch to a df with each pass of a for loop
        for i, batch in enumerate(data.iter_batches(batch_size=batch_size)):
            yield i, total_batches, batch.to_pandas()

    # if data are in a DataFrame, use this code
    elif isinstance(data, pd.DataFrame):

        # num of rows
        total_rows = len(data)

        # num of batches
        total_batches = math.ceil(total_rows / batch_size)

        # generate next i, total_batches, and next df with each pass of a for loop
        # this code doesn't need to convert to a df
        for i in range(total_batches):
            chunk = data.iloc[i * batch_size : (i + 1) * batch_size].copy()
            yield i, total_batches, chunk

    else:

        # alert if the input data weren't in a DataFrame or ParquetFile
        raise TypeError(f"Unsupported input type: {type(data)}. Expected DataFrame, ParquetFile, or file path.")

In [7]:
# define a function to predict SWE in chunks

def predict_basin_SWE(df_dict, ml_models, feature_names, single = False, rename_dict = None, batch_size = 500_000):
    '''
    Predict basin-wise SWE using trained models.

    Args:
    -----
    df_dict (dict): 
        Dictionary of pandas.DataFrames containing basin climate data and engineered features.
        One dictionary can also be input, if a historical dataset or singular DataFrame is input.
    feature_names (list):
            List of feature names models used for training. Used to ensure new DataFrames have the same order.
    ml_models (dict or list):
            Dictionary of trained machine learning models used for prediction.
    single (boolean):
        Assumed to be false, but can be set to true if a single DataFrame is input.
    rename_dict (dict):
        Dictionary of column names to rename features to if needed. 
    batch_size (int):
        Number of rows to stream and predict per batch. Default is 500,000, but can be changed depending on system RAM.
    

    Returns:
    --------
    pd.DataFrame or dict[str, pd.DataFrame]:
        Single DataFrame if single is true, otherwise a dictionary of DataFrames with the MACA model as the key.
    '''

    # set coordinate columns
    coords = ['date', 'latitude', 'longitude']

    # standardize ml models into a list
    ml_model_list = list(ml_models.values())

    # make prism_pq a dict, if applicable
    if single:
        data_dict = {'prism': df_dict}
    else:
        data_dict = df_dict

    # initialize pred dfs
    pred_dfs = {}

    # loop through data_dict    
    for name, data in data_dict.items():
    
        # print start message
        print(f'Beginning SWE prediction for {name}.')

        # initialize chunks and generate batches
        pred_chunks = []
        batch_generator = _get_dataset_batches(data, batch_size)

        # loop through DataFrame in defined batches
        for i, total_batches, df_chunk in batch_generator:
            
            # retain coords
            chunk = df_chunk[coords].copy()

            # rename feature names if needed
            if rename_dict:
                df_chunk = df_chunk.rename(columns = rename_dict)

            # loop through chunk lat and lon, set as float32 to save memory
            for col in ['latitude', 'longitude']:
                if col in chunk.columns:
                    chunk[col] = chunk[col].astype(np.float32)

            # Make sure the df_chunk is filtered to the features in the ml models
            filt_chunk = df_chunk[feature_names]
            
            # Predict across all ensemble models in a single data pass
            model_preds = [model.predict(filt_chunk) for model in ml_model_list]

            # compute ensemble mean
            ens_swe = np.mean(model_preds, axis=0)

            # ensure non-negative SWE
            chunk['swe_pred'] = np.clip(ens_swe, a_min=0, a_max=None).astype(np.float32)

            # append chunk to pred_chunks
            pred_chunks.append(chunk)
            
            # update batch progress
            print(f"[{name}] Predicted batch {i + 1} of {total_batches}.")

            # Prevent memory bloating across batches
            del df_chunk, filt_chunk, model_preds, ens_swe
            gc.collect()

        # Concat batches
        pred_dfs[name] = pd.concat(pred_chunks, ignore_index=True)

        # delete pred chunks to minimize RAM usage
        del pred_chunks
        gc.collect()
        print(f'[{name}] complete. Moving on...')

    # return a pred_df
    if single:
        return pred_dfs['prism']
    else:
        return pred_dfs


In [9]:
prism_col_rename = {
    'prism_ppt_mm': 'ppt_mm', 
    'prism_tmin_c': 'tmin_c', 'prism_tmax_c': 'tmax_c', 'prism_tmean_c': 'tmean_c',
    'prism_tmin_c_roll_3d': 'tmin_c_roll_3d', 'prism_tmax_c_roll_3d': 'tmax_c_roll_3d', 'prism_ppt_mm_roll_3d': 'ppt_mm_roll_3d',
    'prism_tmin_c_roll_7d': 'tmin_c_roll_7d', 'prism_tmax_c_roll_7d': 'tmax_c_roll_7d', 'prism_ppt_mm_roll_7d': 'ppt_mm_roll_7d',
    }

In [10]:
# predict SWE for prism
prism_pred_df = predict_basin_SWE(prism_pq, ml_models, feature_names, single=True, rename_dict=prism_col_rename)

Beginning SWE prediction for prism.
[prism] Predicted batch 1 of 38.
[prism] Predicted batch 2 of 38.
[prism] Predicted batch 3 of 38.
[prism] Predicted batch 4 of 38.
[prism] Predicted batch 5 of 38.
[prism] Predicted batch 6 of 38.
[prism] Predicted batch 7 of 38.
[prism] Predicted batch 8 of 38.
[prism] Predicted batch 9 of 38.
[prism] Predicted batch 10 of 38.
[prism] Predicted batch 11 of 38.
[prism] Predicted batch 12 of 38.
[prism] Predicted batch 13 of 38.
[prism] Predicted batch 14 of 38.
[prism] Predicted batch 15 of 38.
[prism] Predicted batch 16 of 38.
[prism] Predicted batch 17 of 38.
[prism] Predicted batch 18 of 38.
[prism] Predicted batch 19 of 38.
[prism] Predicted batch 20 of 38.
[prism] Predicted batch 21 of 38.
[prism] Predicted batch 22 of 38.
[prism] Predicted batch 23 of 38.
[prism] Predicted batch 24 of 38.
[prism] Predicted batch 25 of 38.
[prism] Predicted batch 26 of 38.
[prism] Predicted batch 27 of 38.
[prism] Predicted batch 28 of 38.
[prism] Predicted bat

In [13]:
# predict SWE for MACA datasets
maca_pred_swe_dfs = predict_basin_SWE(maca_pqs, ml_models, feature_names, batch_size = 3_000_000)

Beginning SWE prediction for IPSL-CM5A-MR.
[IPSL-CM5A-MR] Predicted batch 1 of 4.
[IPSL-CM5A-MR] Predicted batch 2 of 4.
[IPSL-CM5A-MR] Predicted batch 3 of 4.
[IPSL-CM5A-MR] Predicted batch 4 of 4.
[IPSL-CM5A-MR] complete. Moving on...
Beginning SWE prediction for GFDL-ESM2M.
[GFDL-ESM2M] Predicted batch 1 of 4.
[GFDL-ESM2M] Predicted batch 2 of 4.
[GFDL-ESM2M] Predicted batch 3 of 4.
[GFDL-ESM2M] Predicted batch 4 of 4.
[GFDL-ESM2M] complete. Moving on...
Beginning SWE prediction for CNRM-CM5.
[CNRM-CM5] Predicted batch 1 of 4.
[CNRM-CM5] Predicted batch 2 of 4.
[CNRM-CM5] Predicted batch 3 of 4.
[CNRM-CM5] Predicted batch 4 of 4.
[CNRM-CM5] complete. Moving on...
Beginning SWE prediction for CanESM2.
[CanESM2] Predicted batch 1 of 4.
[CanESM2] Predicted batch 2 of 4.
[CanESM2] Predicted batch 3 of 4.
[CanESM2] Predicted batch 4 of 4.
[CanESM2] complete. Moving on...


## Step 4: Format PRISM and MACA as Datasets

In [11]:
# set a multi index
prism_pred_df_multi = prism_pred_df.set_index(['date', 'latitude', 'longitude'])

# convert prism back to Dataset
prism_swe_ds = prism_pred_df_multi.to_xarray()

In [12]:
# check it out
prism_swe_ds

<xarray.Dataset> Size: 124MB
Dimensions:    (date: 7328, latitude: 51, longitude: 83)
Coordinates:
  * date       (date) datetime64[ns] 59kB 1990-10-01 1990-10-02 ... 2020-06-01
  * latitude   (latitude) float32 204B 44.38 44.42 44.46 ... 46.38 46.42 46.46
  * longitude  (longitude) float32 332B -113.9 -113.9 -113.8 ... -110.5 -110.5
Data variables:
    swe_pred   (date, latitude, longitude) float32 124MB nan nan nan ... nan nan

In [14]:
# convert DataFrames to DataSets

# init
maca_pred_swe_dfs_multi = {}
maca_swe_dict = {}

# loop through each df
for model in maca_pred_swe_dfs:
    
    # set a multi index on each df
    maca_pred_swe_dfs_multi[model] = maca_pred_swe_dfs[model].set_index(['date', 'latitude', 'longitude'])
    
    # convert to ds
    maca_swe_dict[model] = maca_pred_swe_dfs_multi[model].to_xarray()

## Step 5: Save PRISM and MACA Datasets to file as NetCDFs

In [17]:
# save prism and maca preds

# set save dir
pred_dir = Path(data_dir, 'ensemble_swe_predictions')
os.makedirs(pred_dir, exist_ok=True)

# save prism
prism_pred_path = Path(pred_dir, 'prism_ML_ensemble_predicted_swe.nc')
prism_swe_ds.to_netcdf(prism_pred_path)

# save maca
# loop through DataSets and save programmatically
for name, ds in maca_swe_dict.items():
    # set path
    save_path = Path(pred_dir, f'maca_{name}_ML_ensemble_midcentury_predicted_swe.nc')
    # save
    ds.to_netcdf(save_path)